# Why Recurrent Neural Networks Are Needed

Numbers for the Note: IMDB review lengths, the cost of zero padding, weight counts, and an ANN tested on shifted reviews. Runs on CPU.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""
import numpy as np, pandas as pd, keras
from sklearn.feature_extraction.text import CountVectorizer

## 1. IMDB review lengths (section 5.1)

In [ ]:
(X_train, y_train), (X_test, y_test) = keras.datasets.imdb.load_data(num_words=10000)
len_train = np.array([len(r) for r in X_train]); len_test = np.array([len(r) for r in X_test])
print(len(len_train), len_train.min(), np.median(len_train), len_train.max())
pd.DataFrame({"length": np.concatenate([len_train, len_test]),
              "split": ["train"] * len(len_train) + ["test"] * len(len_test)}).to_csv("data/imdb_lengths.csv", index=False)

## 2. The cost of padding (sections 5.2 and 5.3)

In [ ]:
longest = len_train.max()
print("share of padding if padded to the longest review:", round(1 - len_train.mean() / longest, 3))
print("share of test reviews longer than 500 words:", round((len_test > 500).mean(), 3))

## 3. Weights: padded ANN layer vs SimpleRNN (sections 5.2 and 6)

In [ ]:
dense = keras.Sequential([keras.Input(shape=(100 * 10000,)), keras.layers.Dense(10)])
rnn = keras.Sequential([keras.Input(shape=(None, 10000)), keras.layers.SimpleRNN(10)])
dense.count_params(), rnn.count_params()

## 4. Bag of words loses the order (section 5.4, Extra)

In [ ]:
cv = CountVectorizer().fit(["dog bites man", "man bites dog"])
cv.get_feature_names_out(), cv.transform(["dog bites man", "man bites dog"]).toarray()

## 5. An ANN tested on shifted reviews (section 5.4)

The ANN reads the first 50 words (1,000-word vocabulary, one-hot inside the network). We test it on the same test reviews pushed right by blank positions. Averaged over 3 seeds.

In [ ]:
V,T=1000,50
(xtr,ytr),(xte,yte)=keras.datasets.imdb.load_data(num_words=V)
def ids(seqs,shift=0):
    X=np.zeros((len(seqs),T),dtype="int32")          # 0 = blank position
    for i,s in enumerate(seqs):
        s=[w for w in s[1:] if w>=3][:T-shift]          # drop the start token and unknown words
        X[i,shift:shift+len(s)]=s
    return X
def model():
    inp=keras.Input(shape=(T,),dtype="int32")
    x=keras.layers.Lambda(lambda z: keras.ops.one_hot(z,V))(inp)   # one-hot inside the network
    x=keras.layers.Flatten()(x)
    x=keras.layers.Dense(16,activation="relu")(x)
    return keras.Model(inp,keras.layers.Dense(1,activation="sigmoid")(x))
res={}
for seed in range(3):
    keras.utils.set_random_seed(seed); m=model()
    m.compile("adam","binary_crossentropy",metrics=["accuracy"])
    m.fit(ids(xtr),ytr,epochs=3,batch_size=256,verbose=0)
    for sh in [0,1,5,10,20]:
        res.setdefault(sh,[]).append(m.evaluate(ids(xte,sh),yte,verbose=0)[1])
for sh,v in res.items(): print(sh, round(float(np.mean(v)),3), np.round(v,3))
